# 01 · Exploratory analysis — Olist orders

Goal: understand coverage, growth, delivery behaviour and the review–delay link before writing findings in `reports/insights.md`.
Run `python src/etl.py` first. Every SQL used here also lives in `sql/`.

In [ ]:
import sqlite3, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
DB = ROOT / "data" / "olist.db"          # built by: python src/etl.py
con = sqlite3.connect(DB)
q = lambda sql: pd.read_sql_query(sql, con)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

## Row counts and order status

In [ ]:
from run_sql import split_named_queries
dq = dict(split_named_queries((ROOT / "sql" / "02_data_quality.sql").read_text()))
display(q(dq["dq_row_counts"]))
display(q(dq["dq_order_status"]))
display(q(dq["dq_customer_identity"]))

## Coverage and growth
The first months (2016) and the last (Sep–Oct 2018) are sparse, so trends use 2017-01 → 2018-08.

In [ ]:
kpi = dict(split_named_queries((ROOT / "sql" / "03_revenue_kpis.sql").read_text()))
monthly = q(kpi["q1_monthly_revenue_growth"])
fig, ax = plt.subplots()
ax.bar(monthly.ym, monthly.revenue, color="#9DB4D3", label="Revenue (BRL)")
ax.plot(monthly.ym, monthly.revenue_3m_avg, color="#1F4E79", lw=2, label="3-month avg")
ax.set_title("Monthly product revenue"); ax.tick_params(axis="x", rotation=60); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
monthly[["ym", "n_orders", "revenue", "mom_growth_pct", "yoy_growth_pct"]].tail(12)

In [ ]:
aov = q(kpi["q2_aov_monthly"])
fig, ax1 = plt.subplots()
ax1.plot(aov.ym, aov.aov, marker="o", label="AOV"); ax1.set_ylabel("AOV (BRL)")
ax2 = ax1.twinx(); ax2.plot(aov.ym, aov.n_orders, color="grey", ls="--", label="orders"); ax2.set_ylabel("orders")
ax1.set_title("Growth: more orders or bigger baskets?"); ax1.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

## Delivery vs estimate

In [ ]:
d = q("""
    SELECT julianday(date(order_delivered_customer_date)) - julianday(date(order_estimated_delivery_date)) AS delay_days,
           julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp) AS actual_days,
           julianday(order_estimated_delivery_date) - julianday(order_purchase_timestamp) AS promised_days
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
""")
print(f"late rate: {(d.delay_days > 0).mean():.2%}   median actual: {d.actual_days.median():.1f} d   median promised: {d.promised_days.median():.1f} d")
fig, ax = plt.subplots()
ax.hist(d.delay_days.clip(-40, 40), bins=80, color="#9DB4D3")
ax.axvline(0, color="crimson", lw=1.5); ax.set_xlabel("delivered − estimated (days, clipped ±40)")
ax.set_title("Most orders arrive well before the estimate; a tail arrives late")
plt.tight_layout(); plt.show()

In [ ]:
dl = dict(split_named_queries((ROOT / "sql" / "06_delivery_delay.sql").read_text()))
lm = q(dl["q5_late_rate_monthly"])
fig, ax = plt.subplots()
ax.plot(lm.ym, lm.late_rate_pct, marker="o", label="late rate %")
ax.plot(lm.ym, lm.late_rate_3m_avg, lw=2, label="3-month avg")
ax.set_title("Late rate by purchase month"); ax.tick_params(axis="x", rotation=60); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## Reviews vs delay — the business cost of lateness

In [ ]:
rv = q(dl["q6_review_vs_delay"])
fig, ax = plt.subplots()
ax.bar(rv.delay_bucket.str[3:], rv.avg_review, color=["#1F4E79"] * 3 + ["crimson"] * (len(rv) - 3))
for i, (s, n) in enumerate(zip(rv.avg_review, rv.n_reviews)):
    ax.text(i, s + 0.05, f"{s:.2f}\n(n={n:,})", ha="center", fontsize=8)
ax.set_ylim(0, 5.3); ax.set_ylabel("avg review"); ax.set_title("Average review score by delivery delay")
ax.tick_params(axis="x", rotation=30); plt.tight_layout(); plt.show()
display(q(dl["q6_review_late_vs_ontime"]))

## Cohort retention (customer_unique_id)

In [ ]:
co = dict(split_named_queries((ROOT / "sql" / "04_cohort_retention.sql").read_text()))
m = q(co["q3_cohort_retention_matrix"]).pivot(index="cohort_month", columns="month_offset", values="retention_pct")
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(m.iloc[:, 1:], aspect="auto", cmap="Blues")
ax.set_xticks(range(m.shape[1] - 1), m.columns[1:]); ax.set_yticks(range(len(m)), m.index)
ax.set_xlabel("months since first purchase"); ax.set_title("Retention % (month 0 = 100%, omitted)")
plt.colorbar(im); plt.tight_layout(); plt.show()
display(q(co["q3_repeat_rate_overall"]))

## Notes for `reports/insights.md`
Write 3–5 findings, each with the number, the query that produced it, and the so-what for the operations manager.